# STAGE 1.5 — warm-start from best (3).pth (mAP 0.8024) and sharpen rank-1
Git-clone workflow: edit code locally -> `git push` -> re-run this notebook (it re-pulls).
All env/X-VLM setup lives in `scripts/kaggle_setup.py` (in the repo).

**Add these Kaggle datasets** (Settings -> Add data), GPU T4, Internet ON:
- `aicity-30k-hard-enhanced` — train_30k_hard_data.tar.zst (webp + jsonl + vitpose + hard_edges)
- `ckpt-30k-hard` — `best (3).pth` + `xvlm_16m_base.th` (+ optional manifest parquet)
- `bbox-dataset` — boxes_30k.json (optional; box GT also comes free from vitpose)


In [ ]:
# [1/7] Clone (or pull) the repo
import os, subprocess, pathlib
REPO_URL = "https://github.com/Khanhhh239/Model_XVLM_Training.git"
ROOT = pathlib.Path("/kaggle/working/Model_XVLM_Training")
if ROOT.exists():
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"], check=False)
else:
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(ROOT)], check=True)
WORK = ROOT / "trainv4"
os.chdir(WORK); print("cwd:", os.getcwd())


In [ ]:
# [2/7] Env + X-VLM source + patches (idempotent). Pin albumentations<2 to match training.
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","albumentations==1.4.18"], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."], check=False)
subprocess.run([sys.executable,"scripts/kaggle_setup.py"], check=True)


In [ ]:
# [3/7] Locate datasets in /kaggle/input (checkpoint base + best(3) + data + boxes)
import glob, os, pathlib, shutil
def find1(pat):
    h = sorted(set(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)))
    return h[0] if h else None

BASE   = find1("xvlm_16m_base.th")                      # X-VLM base (XVLMBackbone builds from this)
BEST3  = find1("best (3).pth") or find1("best*.pth")    # warm-start STARModel (mAP 0.8024)
JSONL  = find1("train_30k_hard.jsonl")
ARCH   = find1("train_30k_hard_data.tar.zst")
VITPOSE= find1("train_30k_hard_vitpose.json")
EDGES  = find1("hard_edges_30k_hard.jsonl")
BOXES  = find1("boxes_30k.json*")
MANI   = find1("manifest_30k_hard_enhanced.parquet")    # optional prebuilt manifest

if not JSONL and ARCH:                                  # extract the .tar.zst once
    dst = "/kaggle/working/ext"; os.makedirs(dst, exist_ok=True)
    if not glob.glob(f"{dst}/**/train_30k_hard.jsonl", recursive=True):
        print("extracting", ARCH, "...")
        if shutil.which("zstd"):
            os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {dst}")
        else:
            import zstandard, tarfile
            with open(ARCH,"rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tf: tf.extractall(dst)
    JSONL   = JSONL   or find1("train_30k_hard.jsonl")
    VITPOSE = VITPOSE or find1("train_30k_hard_vitpose.json")
    EDGES   = EDGES   or find1("hard_edges_30k_hard.jsonl")

WEBP = None
for c in glob.glob("/kaggle/input/**/train_webp", recursive=True)+glob.glob("/kaggle/working/**/train_webp", recursive=True):
    if os.path.isdir(c): WEBP = c; break
assert BASE and BEST3 and (MANI or (JSONL and WEBP)), f"missing: BASE={BASE} BEST3={BEST3} MANI={MANI} JSONL={JSONL} WEBP={WEBP}"
print("BASE",BASE,"\nBEST3",BEST3,"\nMANI",MANI,"\nJSONL",JSONL,"\nWEBP",WEBP,"\nVITPOSE",VITPOSE,"\nEDGES",EDGES,"\nBOXES",BOXES)


In [ ]:
# [4/7] Build the manifest if not provided (videos -> train/valb distractor split)
import subprocess, sys, pathlib
MANIFEST = MANI
if not MANIFEST:
    MANIFEST = "/kaggle/working/manifest_30k_hard_enhanced.parquet"
    subprocess.run([sys.executable,"scripts/build_stage1_manifest.py",
                    "--jsonl",JSONL,"--vitpose",VITPOSE or "","--image-root",WEBP,
                    "--out",MANIFEST], check=True)
print("manifest:", MANIFEST)


In [ ]:
# [5/7] Write the runtime config (override Stage-1.5 template paths with the located files)
import yaml
cfg = yaml.safe_load(open("configs/stage1_warmstart_best3.yaml"))
cfg["data"]["manifest"]        = MANIFEST
cfg["data"]["image_root"]      = WEBP or cfg["data"]["image_root"]
cfg["data"]["vitpose_json"]    = VITPOSE
cfg["data"]["hard_edges_json"] = EDGES
cfg["data"]["boxes_json"]      = BOXES                 # None is fine (vitpose box used)
cfg["model"]["checkpoint"]     = BASE                  # X-VLM base; best(3) loaded via --init-from
yaml.safe_dump(cfg, open("configs/_runtime.yaml","w"), sort_keys=False)
print(open("configs/_runtime.yaml").read())


In [ ]:
# [6/7] Sanity: overfit one batch (loss must drop fast). ~5-10 min.
import subprocess, sys
subprocess.run([sys.executable,"scripts/train.py","--config","configs/_runtime.yaml",
                "--init-from",BEST3,"--overfit-one-batch"], check=True)


In [ ]:
# [7/7] Full training: warm-start best(3).pth, save best.pth, revert if mAP < baseline.
# Resumes from last.pth across the 12h limit. ANCE re-mine + hard_edges run each epoch.
import subprocess, sys, os
last = "/kaggle/working/out_stage1_best3/last.pth"
cmd = [sys.executable,"scripts/train.py","--config","configs/_runtime.yaml","--max-hours","11.0"]
cmd += ["--resume", last] if os.path.exists(last) else ["--init-from", BEST3]
subprocess.run(cmd, check=True)
import shutil, pathlib
best = pathlib.Path("/kaggle/working/out_stage1_best3/best.pth")
if best.exists():
    shutil.copy(best, "/kaggle/working/stage15_best.pth")
    print("saved /kaggle/working/stage15_best.pth — download from the Output tab")
